In [3]:


# AEGIS-X COMPLETE ONE-CELL LAUNCHER
# No Gradio, no emojis, automatic free-port selection

import os, sys, time, socket, subprocess, textwrap

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "streamlit", "numpy", "pandas", "scikit-learn",
    "networkx", "plotly", "pyngrok"
])

APP = r'''
import json
import numpy as np
import pandas as pd
import networkx as nx
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

st.set_page_config(page_title="AEGIS-X", layout="wide")

st.markdown("""
<style>
.stApp {background:linear-gradient(135deg,#07111f,#10243c);color:#e8f1ff}
[data-testid="stMetric"] {background:#142943;padding:15px;border:1px solid #2b4c70;border-radius:12px}
h1,h2,h3 {color:#d9eaff}
</style>
""", unsafe_allow_html=True)

FEATURES = [
    "cpu_percent","memory_percent","latency_ms",
    "error_rate","requests_per_sec","queue_depth"
]

@st.cache_data
def make_data(n=6000, seed=42):
    rng = np.random.default_rng(seed)
    cpu = rng.beta(2,2.2,n)*100
    memory = rng.beta(2.2,2,n)*100
    latency = rng.lognormal(4,.65,n)
    errors = rng.beta(1,18,n)*100
    requests = rng.lognormal(5,.7,n)
    queue = rng.exponential(22,n)

    overloaded = (cpu>78)|(memory>86)
    latency[overloaded] *= rng.uniform(1.3,3,overloaded.sum())
    errors[overloaded] += rng.uniform(1,12,overloaded.sum())
    queue[overloaded] *= rng.uniform(1.5,4,overloaded.sum())

    latency=np.clip(latency,1,5000)
    errors=np.clip(errors,0,100)
    queue=np.clip(queue,0,1000)

    signal=((cpu>88).astype(int)+(memory>92).astype(int)
            +(latency>500).astype(int)+(errors>8).astype(int)
            +(queue>130).astype(int))
    probability=1/(1+np.exp(-1.3*(signal-1.6)))
    labels=(rng.random(n)<probability).astype(int)

    return pd.DataFrame({
        "cpu_percent":cpu,"memory_percent":memory,
        "latency_ms":latency,"error_rate":errors,
        "requests_per_sec":requests,"queue_depth":queue,
        "failure_risk_label":labels
    })

@st.cache_resource
def train():
    data=make_data()
    X=data[FEATURES]
    y=data["failure_risk_label"]
    Xtr,Xte,ytr,yte=train_test_split(
        X,y,test_size=.25,random_state=42,stratify=y
    )
    rf=RandomForestClassifier(
        n_estimators=160,max_depth=12,min_samples_leaf=3,
        class_weight="balanced",random_state=42,n_jobs=-1
    )
    rf.fit(Xtr,ytr)
    iso=IsolationForest(
        n_estimators=120,contamination=.06,random_state=42
    )
    iso.fit(Xtr)
    pred=rf.predict(Xte)
    prob=rf.predict_proba(Xte)[:,1]
    metrics={
        "Accuracy":accuracy_score(yte,pred),
        "Precision":precision_score(yte,pred,zero_division=0),
        "Recall":recall_score(yte,pred,zero_division=0),
        "F1 Score":f1_score(yte,pred,zero_division=0),
        "ROC-AUC":roc_auc_score(yte,prob)
    }
    return data,rf,iso,metrics,yte,pred

data,model,iso,metrics,y_test,predictions=train()

def make_graph():
    G=nx.DiGraph()
    services={
        "Load Balancer":("Edge",2,.025),
        "API Gateway":("Gateway",2,.035),
        "API Service":("Application",3,.080),
        "Database":("Data",2,.120),
        "Cache":("Data",2,.040),
        "AI Inference":("AI",2,.150),
        "Message Queue":("Messaging",2,.055)
    }
    for name,(tier,replicas,cost) in services.items():
        G.add_node(name,tier=tier,replicas=replicas,cost=cost)
    G.add_edges_from([
        ("Load Balancer","API Gateway"),
        ("API Gateway","API Service"),
        ("API Service","Database"),
        ("API Service","Cache"),
        ("API Service","AI Inference"),
        ("API Service","Message Queue"),
        ("Message Queue","Database")
    ])
    return G

GRAPH=make_graph()

def topology(G,failed):
    pos=nx.spring_layout(G,seed=17)
    ex,ey=[],[]
    for u,v in G.edges():
        ex += [pos[u][0],pos[v][0],None]
        ey += [pos[u][1],pos[v][1],None]
    edge=go.Scatter(
        x=ex,y=ey,mode="lines",
        line=dict(width=1.5,color="#597899"),hoverinfo="none"
    )
    nodes=list(G.nodes())
    node=go.Scatter(
        x=[pos[n][0] for n in nodes],
        y=[pos[n][1] for n in nodes],
        mode="markers+text",text=nodes,textposition="bottom center",
        marker=dict(
            size=25,
            color=["#ef4444" if n==failed else "#22c55e" for n in nodes],
            line=dict(width=1,color="white")
        ),
        hovertext=[
            f"{n}<br>Tier: {G.nodes[n]['tier']}<br>Replicas: {G.nodes[n]['replicas']}"
            for n in nodes
        ],hoverinfo="text"
    )
    fig=go.Figure([edge,node])
    fig.update_layout(
        title="Cloud Service Dependency Graph",height=480,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",font_color="white",
        showlegend=False,xaxis=dict(visible=False),
        yaxis=dict(visible=False)
    )
    return fig

def simulate(failed,traffic,severity,strategy,seed=42):
    rng=np.random.default_rng(seed)
    n=120
    cpu=rng.normal(48,9,n)
    memory=rng.normal(57,7,n)
    latency=rng.lognormal(4,.25,n)
    errors=rng.beta(1,30,n)*100
    requests=rng.normal(450,55,n)*traffic
    queue=rng.exponential(15,n)

    if failed!="None":
        effects={
            "Load Balancer":(10,100),"API Gateway":(22,250),
            "API Service":(22,250),"Database":(15,180),
            "Cache":(10,120),"AI Inference":(12,300),
            "Message Queue":(10,180)
        }
        cb,lb=effects[failed]
        cpu[35:75]+=cb*severity
        memory[35:75]+=9*severity
        latency[35:75]+=lb*severity
        errors[35:75]+=4.5*severity
        queue[35:75]+=45*severity

        effects_by_strategy={
            "Restart service":(.91,.72,.55,.60),
            "Scale replicas":(.70,.68,.75,.62),
            "Reroute traffic":(.82,.76,.60,.70),
            "Circuit breaker":(1,.65,.72,.45)
        }
        if strategy in effects_by_strategy:
            a,b,c,d=effects_by_strategy[strategy]
            cpu[60:]*=a
            latency[60:]*=b
            errors[60:]*=c
            queue[60:]*=d

    cpu=np.clip(cpu*(.8+.2*traffic),1,100)
    memory=np.clip(memory,1,100)
    latency=np.clip(latency*(.8+.35*(traffic-1)),1,5000)
    errors=np.clip(errors,0,100)
    requests=np.clip(requests,0,None)
    queue=np.clip(queue*traffic,0,1000)

    df=pd.DataFrame({
        "time_step":np.arange(n),"cpu_percent":cpu,
        "memory_percent":memory,"latency_ms":latency,
        "error_rate":errors,"requests_per_sec":requests,
        "queue_depth":queue
    })
    df["risk_percent"]=model.predict_proba(df[FEATURES])[:,1]*100
    df["anomaly"]=iso.predict(df[FEATURES])==-1

    hourly=sum(a["cost"]*a["replicas"] for _,a in GRAPH.nodes(data=True))
    multiplier=1.3 if strategy=="Scale replicas" else 1.0
    summary={
        "Mean CPU (%)":float(cpu.mean()),
        "Peak CPU (%)":float(cpu.max()),
        "Mean memory (%)":float(memory.mean()),
        "Mean latency (ms)":float(latency.mean()),
        "Peak latency (ms)":float(latency.max()),
        "Mean error rate (%)":float(errors.mean()),
        "Mean queue depth":float(queue.mean()),
        "Mean failure risk (%)":float(df.risk_percent.mean()),
        "Peak failure risk (%)":float(df.risk_percent.max()),
        "Anomaly observations":int(df.anomaly.sum()),
        "Estimated monthly cost ($)":float(hourly*multiplier*24*30),
        "Illustrative availability (%)":float(np.clip(
            99.99-errors.mean()*.08-latency.mean()/4000,90,99.99
        ))
    }
    return df,summary

st.sidebar.title("AEGIS-X")
st.sidebar.caption("Cloud Reliability Intelligence")
failed=st.sidebar.selectbox(
    "Inject a service failure",["None"]+list(GRAPH.nodes())
)
traffic=st.sidebar.slider("Traffic multiplier",.5,3.0,1.0,.1)
severity=st.sidebar.slider("Failure severity",.5,2.0,1.0,.1)
strategy=st.sidebar.selectbox(
    "Recovery strategy",
    ["No intervention","Restart service","Scale replicas",
     "Reroute traffic","Circuit breaker"]
)
telemetry,summary=simulate(failed,traffic,severity,strategy)

st.title("AEGIS-X")
st.subheader("AI-Powered Cloud Intelligence and Self-Healing Infrastructure")
st.write(
    "Predict infrastructure risks, detect anomalous telemetry, "
    "visualize service dependencies and compare recovery strategies."
)
st.warning(
    "This is a research simulation. Telemetry, labels, recovery effects, "
    "costs and availability estimates are synthetic. No live cloud resources "
    "are accessed or modified."
)

cols=st.columns(5)
kpis=[
    ("Mean CPU",f"{summary['Mean CPU (%)']:.1f}%"),
    ("Mean latency",f"{summary['Mean latency (ms)']:.1f} ms"),
    ("Failure risk",f"{summary['Mean failure risk (%)']:.1f}%"),
    ("Anomaly observations",str(summary["Anomaly observations"])),
    ("Availability estimate",f"{summary['Illustrative availability (%)']:.3f}%")
]
for col,(label,value) in zip(cols,kpis):
    col.metric(label,value)

tab1,tab2,tab3,tab4,tab5=st.tabs([
    "Operations","Recovery Lab","AI Model Lab","Data Explorer","Documentation"
])

with tab1:
    left,right=st.columns([1.2,1])
    with left:
        st.plotly_chart(topology(GRAPH,failed),use_container_width=True)
    with right:
        st.subheader("Infrastructure status")
        if failed=="None":
            st.success("No failure injected.")
        else:
            st.error(f"Simulated failure: {failed}")
        st.write(f"Traffic multiplier: {traffic:.1f}x")
        st.write(f"Failure severity: {severity:.1f}x")
        st.write(f"Recovery strategy: {strategy}")
        if failed!="None":
            affected=list(nx.descendants(GRAPH,failed))
            st.write("Downstream dependencies:")
            for service in affected:
                st.write("- "+service)
            if not affected:
                st.write("No downstream nodes in this graph.")
        resource=pd.DataFrame({
            "Resource":["CPU","Memory"],
            "Utilization (%)":[summary["Mean CPU (%)"],summary["Mean memory (%)"]]
        })
        st.plotly_chart(px.bar(
            resource,x="Resource",y="Utilization (%)",
            range_y=[0,100],title="Mean resource utilization"
        ),use_container_width=True)

    metric=st.selectbox("Telemetry metric",[
        "cpu_percent","memory_percent","latency_ms","error_rate",
        "requests_per_sec","queue_depth","risk_percent"
    ])
    st.plotly_chart(px.line(
        telemetry,x="time_step",y=metric,
        title=metric.replace("_"," ").title()
    ),use_container_width=True)
    st.plotly_chart(px.area(
        telemetry,x="time_step",y="risk_percent",
        title="Predicted failure risk over simulated time"
    ),use_container_width=True)

with tab2:
    st.subheader("Recovery strategy experiments")
    if failed=="None":
        st.info("Select a service failure in the sidebar to compare recovery.")
    else:
        rows=[]
        options=["No intervention","Restart service","Scale replicas",
                 "Reroute traffic","Circuit breaker"]
        for recovery in options:
            _,result=simulate(failed,traffic,severity,recovery)
            rows.append({
                "Strategy":recovery,
                "Mean CPU (%)":result["Mean CPU (%)"],
                "Mean latency (ms)":result["Mean latency (ms)"],
                "Mean errors (%)":result["Mean error rate (%)"],
                "Mean risk (%)":result["Mean failure risk (%)"],
                "Monthly cost ($)":result["Estimated monthly cost ($)"],
                "Availability estimate (%)":result["Illustrative availability (%)"]
            })
        comparison=pd.DataFrame(rows)
        st.dataframe(comparison,use_container_width=True)
        st.plotly_chart(px.bar(
            comparison,x="Strategy",y="Mean risk (%)",color="Strategy",
            title="Predicted risk by recovery strategy"
        ),use_container_width=True)
        st.plotly_chart(px.bar(
            comparison,x="Strategy",y="Monthly cost ($)",color="Strategy",
            title="Illustrative monthly cost"
        ),use_container_width=True)
        best=comparison.loc[comparison["Mean risk (%)"].idxmin()]
        st.info(
            f"Lowest simulated mean risk: {best['Strategy']} "
            f"({best['Mean risk (%)']:.2f}%). Not a production guarantee."
        )

with tab3:
    st.subheader("Model evaluation on synthetic test data")
    metric_cols=st.columns(5)
    for col,(name,value) in zip(metric_cols,metrics.items()):
        col.metric(name,f"{value:.3f}")
    st.caption(
        "Labels are synthetic and rule-generated. These metrics do not "
        "establish real-world cloud failure prediction accuracy."
    )
    cm=confusion_matrix(y_test,predictions,labels=[0,1])
    st.plotly_chart(px.imshow(
        cm,text_auto=True,
        x=["Predicted normal","Predicted risk"],
        y=["Actual normal","Actual risk"],
        title="Confusion matrix",color_continuous_scale="Blues"
    ),use_container_width=True)
    importance=pd.DataFrame({
        "Feature":FEATURES,"Importance":model.feature_importances_
    }).sort_values("Importance")
    st.plotly_chart(px.bar(
        importance,x="Importance",y="Feature",orientation="h",
        title="Random Forest feature importance"
    ),use_container_width=True)

    st.subheader("Custom telemetry prediction")
    a,b,c=st.columns(3)
    with a:
        cpu_in=st.slider("CPU (%)",0,100,85)
        mem_in=st.slider("Memory (%)",0,100,80)
    with b:
        latency_in=st.number_input("Latency (ms)",1.0,5000.0,450.0)
        errors_in=st.number_input("Error rate (%)",0.0,100.0,5.0)
    with c:
        requests_in=st.number_input("Requests per second",0.0,100000.0,500.0)
        queue_in=st.number_input("Queue depth",0.0,10000.0,120.0)

    custom=pd.DataFrame([{
        "cpu_percent":cpu_in,"memory_percent":mem_in,
        "latency_ms":latency_in,"error_rate":errors_in,
        "requests_per_sec":requests_in,"queue_depth":queue_in
    }])[FEATURES]
    risk=model.predict_proba(custom)[0,1]*100
    anomaly=iso.predict(custom)[0]==-1
    r1,r2,r3=st.columns(3)
    r1.metric("Predicted failure risk",f"{risk:.2f}%")
    r2.metric("Anomaly detected","Yes" if anomaly else "No")
    r3.metric("Risk category","Higher" if risk>=50 else "Lower")

with tab4:
    st.subheader("Simulated telemetry")
    st.dataframe(telemetry,use_container_width=True)
    st.download_button(
        "Download telemetry CSV",
        telemetry.to_csv(index=False).encode("utf-8"),
        "aegis_x_telemetry.csv","text/csv"
    )
    summary_df=pd.DataFrame([
        {"Metric":k,"Value":v} for k,v in summary.items()
    ])
    st.subheader("Scenario summary")
    st.dataframe(summary_df,use_container_width=True)
    st.download_button(
        "Download summary JSON",
        json.dumps(summary,indent=2).encode("utf-8"),
        "aegis_x_summary.json","application/json"
    )
    st.subheader("Synthetic training dataset")
    st.dataframe(data.head(100),use_container_width=True)
    st.download_button(
        "Download training data CSV",
        data.to_csv(index=False).encode("utf-8"),
        "aegis_x_training_data.csv","text/csv"
    )

with tab5:
    st.subheader("System architecture")
    st.code("""
AEGIS-X
|
+-- Synthetic telemetry generator
|   +-- CPU, memory, latency, errors
|   +-- Requests and queue depth
+-- AI intelligence
|   +-- Random Forest classifier
|   +-- Isolation Forest anomaly detector
+-- Dependency intelligence
|   +-- Directed service graph
|   +-- Downstream impact analysis
+-- Recovery experiments
|   +-- Restart, scaling, rerouting, circuit breaker
+-- Streamlit dashboard
    +-- Operations, recovery, model lab, data exports
""",language="text")
    st.subheader("Technology stack")
    st.markdown("""
    - Python: application logic
    - NumPy and Pandas: data processing
    - Scikit-learn: machine learning
    - NetworkX: service dependency graph
    - Plotly: interactive visualizations
    - Streamlit: dashboard
    """)
    st.subheader("Limitations")
    st.markdown("""
    1. Telemetry and labels are synthetic.
    2. Recovery effects are simplified assumptions.
    3. Cost estimates are illustrative, not cloud-provider prices.
    4. Availability is an estimate, not measured uptime.
    5. No live cloud account is accessed.
    6. Production use requires real telemetry, validation, security,
       audit logging and human approval for remediation.
    """)

st.divider()
st.caption("AEGIS-X | Research prototype | Synthetic simulation")
'''

# Save app.py
with open("app.py", "w", encoding="utf-8") as f:
    f.write(textwrap.dedent(APP))

print("Application saved to app.py")

# Find an available port to avoid the port 8501 conflict
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    sock.bind(("0.0.0.0", 0))
    PORT = sock.getsockname()[1]

print("Selected port:", PORT)

log = open("aegis_streamlit.log", "w", encoding="utf-8")
process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port", str(PORT),
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--browser.gatherUsageStats", "false"
    ],
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(6)

if process.poll() is not None:
    log.close()
    with open("aegis_streamlit.log", encoding="utf-8") as f:
        print(f.read()[-10000:])
    raise RuntimeError("Streamlit failed to start. See the log above.")

print("Streamlit started.")

# Attempt a public tunnel for Colab
try:
    from pyngrok import ngrok
    token = os.environ.get("NGROK_AUTH_TOKEN")
    if token:
        ngrok.set_auth_token(token)
    url = ngrok.connect(PORT, "http")
    print("Dashboard URL:")
    print(url)
    print("Keep this cell running while using the dashboard.")
except Exception as exc:
    print("Streamlit is running on port", PORT)
    print("Local URL: http://localhost:" + str(PORT))
    print("For Colab, configure ngrok if a public URL is needed.")
    print("Tunnel message:", exc)


Application saved to app.py
Selected port: 38309
Streamlit started.


ERROR:pyngrok.process.ngrok:t=2026-10-10T06:21:12+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-10-10T06:21:12+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


Streamlit is running on port 38309
Local URL: http://localhost:38309
For Colab, configure ngrok if a public URL is needed.
Tunnel message: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.
